# CoChem-BASE: Master Environment Orchestrator

Welcome to the **CoChem-BASE** initialization matrix. This environment replaces all legacy Docker and DevContainer layers, allowing you to natively provision your exact Interaction and Calculation hardware profiles.

### 🚀 Quick Start Instructions
1. Click the code cell below and press `Shift + Enter` to Setup the Silo and Environment Paths.
2. Select your newly created `cochem_base_silo` Kernel when prompted.
3. Once selected, run the final cell to render the Matrix Dashboard.


## 💾 Silo Setup & Artifact Registry Configuration

**Purpose:**
To establish a dedicated, reproducible computational environment (Silo) and configure a persistent local directory for storing generated chemistry artifacts.

**Instructions:**
- Run this code cell below by clicking it and pressing `Shift + Enter`.
- **If** you have a preferred directory for storing molecular structures, job outputs, and logs, **then** enter it into the text box.
- **If** you prefer the default configuration, **then** simply leave the default path (which creates a `CoChem_Artifacts` folder in your home directory).
- Click the green **Set Path & Build Silo** button to initiate the build process.
- **If** the build succeeds, **then** a green success box will appear containing a link to select your new kernel.
- **If** the build fails, **then** ensure that Conda/Miniconda is installed and correctly configured in your system PATH.

**Didactic Breakdown:**
In computational chemistry and chemoinformatics, having exact software environments is absolutely critical. Slight differences in dependency versions (like OpenBabel, RDKit, or ASE) can lead to irreproducible single-point energies, broken molecular trajectory visualizations, or incompatible quantum mechanical (QM) properties. 

This cell programmatically invokes a background process that utilizes Conda to construct a completely isolated virtual environment (termed the *"cochem_base_silo"*). By doing this, we guarantee that all downstream QM/MM solvers and interaction tools are operating within strictly deterministic, identical boundaries. This completely safeguards the scientific integrity and reproducibility of your computational experiments.

In [ ]:
import json
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as widgets

print("=======================================================")
print(" ⚙️ CoChem-BASE: Artifact & Silo Registry Configuration")
print("=======================================================\n")

path_input = widgets.Text(
    value=str(Path.home() / "CoChem_Artifacts"),
    placeholder="Enter absolute path",
    description="Storage Path:",
    layout=widgets.Layout(width="80%")
)
set_btn = widgets.Button(description="Set Path & Build Silo", button_style="success", layout=widgets.Layout(width="200px"))
output = widgets.Output()

def on_click(b):
    set_btn.disabled = True
    set_btn.description = "Building Silo..."
    with output:
        output.clear_output()
        cfg = {"artifact_dir": path_input.value}
        cfg_path = Path.cwd() / ".cochem_env.json"
        with open(cfg_path, "w") as f:
            json.dump(cfg, f)
        print(f"✅ Saved Artifact registry path to: {cfg_path}")
        print(f"⚙️ Target Artifact Director: {path_input.value}\n")
        print("▶️ Handing off to Silo creation agent...\n")
        
        log_output = widgets.Textarea(value='', disabled=True, layout=widgets.Layout(height='250px', width='100%'))
        display(log_output)
        
        def run_silo():
            import subprocess
            import sys
            try:
                proc = subprocess.Popen(
                    [sys.executable, "setup/cochem_base_silo_setup.py"], 
                    stdout=subprocess.PIPE, 
                    stderr=subprocess.STDOUT, 
                    text=True, 
                    encoding="utf-8",
                    bufsize=1
                )
                for line in iter(proc.stdout.readline, ''):
                    log_output.value += line
                proc.wait()
                if proc.returncode == 0:
                    html_str = """
                        <div style="padding: 10px; background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 5px; color: #155724; margin-top: 15px;">
                            <b>✅ Conda Silo Provisioned!</b><br><br>
                            Please click the secure link below to trigger the VS Code Kernel Selector. <br>
                            Select the newly created <b>cochem_base_silo</b> environment, wait 2 seconds for it to attach, and then execute the UI Matrix block below.<br><br>
                            <a href="command:notebook.selectKernel" style="font-size: 16px; font-weight: bold; padding: 5px 10px; background-color: #155724; color: white; text-decoration: none; border-radius: 3px;">🔄 Select cochem_base_silo Kernel</a>
                        </div>
                    """
                    output.append_display_data(HTML(html_str))
                else:
                    log_output.value += f"\n❌ Silo build failed with code {proc.returncode}\n"
            except Exception as e:
                log_output.value += f"Error launching script: {e}\n"
            finally:
                set_btn.disabled = False
                set_btn.description = "Set Path & Build Silo"
                
        import threading
        threading.Thread(target=run_silo, daemon=True).start()

set_btn.on_click(on_click)
display(widgets.VBox([widgets.HBox([path_input, set_btn]), output]))


## 🎛️ CoChem-BASE Interactive Matrix Dashboard

**Purpose:**
To launch the primary graphical user interface used to route computational chemistry tasks, configure execution environments, and integrate external chemistry software (like ORCA).

**Instructions:**
- **If** you just provisioned the Silo in the previous cell, **then** you MUST click the kernel link provided in the success message (or use the kernel selector in the top right of VS Code) to switch your active Python kernel to `cochem_base_silo`.
- Wait exactly 2 seconds for the kernel to attach.
- Execute the code cell below by clicking it and pressing `Shift + Enter`.
- **If** an error stating "UNITY dashboard missing" appears, **then** verify you are running this notebook from the root directory of the CoChem-BASE repository.
- **If** the dashboard successfully loads, **then** you may proceed to configure your interaction and calculation hardware profiles directly via the UI.

**Didactic Breakdown:**
Executing advanced chemical computations often requires orchestrating incredibly complex workflows across diverse hardware architectures (e.g., transitioning jobs between local graphical workstations, Windows Subsystem for Linux (WSL), and High-Performance Computing (HPC) clusters).

This cell bridges the gap between high-level Jupyter notebook interaction and low-level subprocess execution. It dynamically maps and injects a custom Python-based GUI into memory, preventing pollution of the system path while surfacing a robust dashboard. This architecture empowers researchers to intuitively configure molecular dynamics simulations, electronic structure jobs, and thermodynamic modeling parameters without being forced to manually edit complex, error-prone shell scripts or JSON configuration files.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
from IPython.display import display, HTML

# Bootstrap environment based on UI selection
cfg_path = Path.cwd() / ".cochem_env.json"
if cfg_path.exists():
    import json
    try:
        with open(cfg_path, "r") as f:
            os.environ["COCHEM_ARTIFACT_DIR"] = json.load(f).get("artifact_dir", "")
    except:
        pass

# 1. Establish strict pathing to bypass shell scripts
REPO_ROOT = Path.cwd()
dashboard_script = REPO_ROOT / "interfaces" / "cochem_unity_installer_dashboard.py"

# 2. Enforce File-Integrity Checks
if not dashboard_script.exists():
    display(HTML(f"<b style=\"color:red;\">❌ FATAL: UNITY dashboard missing at {dashboard_script}</b>"))
else:
    # 3. Dynamically map the GUI into memory without sys.path pollution
    spec = importlib.util.spec_from_file_location("cochem_unity_installer_dashboard", str(dashboard_script))
    dashboard_module = importlib.util.module_from_spec(spec)
    sys.modules["cochem_unity_installer_dashboard"] = dashboard_module
    spec.loader.exec_module(dashboard_module)
    
    # 4. Render the CoChem-BASE UI Matrix
    installer = dashboard_module.SynapInstallerGUI()
    display(installer.build_ui())
